In [ ]:
"""
Buy Bot — Telegram Poller → IBKR Paper Trading → MongoDB
Listens for NEW signals, validates EMA, places limit buy, records trade.

Fix: uses StringSession (no SQLite file lock) with the session string
persisted to SESSION_FILE so auth survives restarts without re-OTP.
Fix 2: forces UTF-8 on all log handlers so Windows cp1252 never sees emoji.
"""

import asyncio
import logging
import os
import sys
import threading
import re
from datetime import datetime, timezone
import random

import pandas as pd
from pymongo import MongoClient

from telethon import TelegramClient
from telethon.sessions import StringSession
from telethon.errors import FloodWaitError, UserAlreadyParticipantError
from telethon.tl.functions.messages import ImportChatInviteRequest

from ib_insync import *

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# ─── Logging  (UTF-8 safe: works in Jupyter AND Windows terminal) ─────────────
class _Utf8SafeStreamHandler(logging.StreamHandler):
    """
    Never raises UnicodeEncodeError.
    Works in Jupyter (no fileno), Windows cp1252, and normal shells.
    """
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass  # Jupyter OutStream has no reconfigure -- that is fine


def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")

    sh = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)

    fh = logging.FileHandler("buy_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)

    logger = logging.getLogger("buy_bot")
    logger.setLevel(logging.INFO)
    # Clear so re-running the cell in Jupyter does not duplicate output
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ─── Telegram Credentials ─────────────────────────────────────────────────────
API_ID        = 34025130
API_HASH      = "e5e514ebcb8a15042feadfecd7a15cfe"
PHONE         = "+18477675748"
INVITE_LINK   = "https://t.me/+M5HRCxZycqJmOTg5"
POLL_INTERVAL = 20   # seconds

# StringSession avoids SQLite file-lock issues entirely.
# On first run it asks for your OTP and saves the session string here.
SESSION_FILE  = "buy_bot_session.txt"

# ─── IBKR Paper Trading ───────────────────────────────────────────────────────
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497        # TWS paper=7497 | IB Gateway paper=4002
IBKR_CLIENT_ID = random.randint(1, 1000)

# ─── Trade Rules ──────────────────────────────────────────────────────────────
ORDER_SIZE = 10   # fixed shares

# ─── MongoDB ──────────────────────────────────────────────────────────────────
mongo_client              = MongoClient("mongodb://localhost:27017/")
db                        = mongo_client["brkout_database"]
trades_collection         = db["trades"]
failed_entries_collection = db["failed_bad_entries"]
ticker_collection         = db["tickers"]
ticker_trade_collection   = db["tickers_trade"]


# ══════════════════════════════════════════════════════════════════════════════
# Telegram Parsers
# ══════════════════════════════════════════════════════════════════════════════
def extract_ticker(message_text: str) -> str | None:
    text = message_text.strip()
    m = re.match(r'^\[.*?\]\s+([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    m = re.match(r'^([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    return None


def parse_structured_message(text: str) -> dict | None:
    result = {"symbol": None, "status": None}
    for line in text.splitlines():
        line = line.strip()
        if re.match(r"(?i)^ticker\s*:", line):
            result["symbol"] = line.split(":", 1)[1].strip().upper()
        if re.match(r"(?i)^status\s*:", line):
            result["status"] = line.split(":", 1)[1].strip()
    if result["symbol"] and result["status"]:
        return result
    return None


def is_new_signal(status_text: str) -> bool:
    return "new" in status_text.lower()


# ══════════════════════════════════════════════════════════════════════════════
# Safe print — strips / replaces characters the console can't handle
# ══════════════════════════════════════════════════════════════════════════════
def safe_print(text: str):
    """
    Print to stdout, replacing any unencodable characters with '?'.
    Works in Jupyter (no .buffer) and normal terminals.
    """
    try:
        # Normal terminal: write bytes directly to avoid codec issues
        encoded = text.encode(sys.stdout.encoding or "utf-8", errors="replace")
        sys.stdout.buffer.write(encoded + b"\n")
        sys.stdout.buffer.flush()
    except AttributeError:
        # Jupyter: OutStream has no .buffer — just print, it handles encoding
        print(text.encode("utf-8", errors="replace").decode("utf-8", errors="replace"))
    except Exception:
        print(text)  # last resort


# ══════════════════════════════════════════════════════════════════════════════
# IBKR Client (Buy-side only)
# ══════════════════════════════════════════════════════════════════════════════
class IBKRBuyClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()

    def connect(self):
        self.ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    # ── Checks ────────────────────────────────────────────────────────────────
    def has_open_position(self, symbol: str) -> bool:
        return any(
            p.contract.symbol == symbol and p.position != 0
            for p in self.ib.positions()
        )

    def has_pending_order(self, symbol: str) -> bool:
        return any(t.contract.symbol == symbol for t in self.ib.openTrades())

    # ── Market Data ───────────────────────────────────────────────────────────
    def get_ask_price(self, symbol: str) -> float | None:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "", False, False)
        self.ib.sleep(2)
        self.ib.cancelMktData(contract)
        if ticker.ask and ticker.ask > 0:
            return float(ticker.ask)
        if ticker.last and ticker.last > 0:
            return float(ticker.last)
        return None

    def get_1min_bars(self, symbol: str, n: int = 70) -> pd.DataFrame | None:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        raw = self.ib.reqHistoricalData(
            contract,
            endDateTime="",
            durationStr="5 D",
            barSizeSetting="5 min",
            whatToShow="TRADES",
            useRTH=True,
        )
        if not raw:
            return None
        df = pd.DataFrame([
            {"high": b.high, "low": b.low, "close": b.close}
            for b in raw
        ]).tail(n).reset_index(drop=True)
        return df

    def ema_bullish(self, symbol: str) -> bool:
        try:
            df = self.get_1min_bars(symbol, n=70)
            if df is None or len(df) < 50:
                log.warning(f"{symbol}: not enough bars for EMA")
                return False
            ema21 = df["close"].ewm(span=21, adjust=False).mean().iloc[-1]
            ema50 = df["close"].ewm(span=50, adjust=False).mean().iloc[-1]
            log.info(f"{symbol}: EMA21={ema21:.4f}  EMA50={ema50:.4f}  bullish={ema21 > ema50}")
            return bool(ema21 > ema50)
        except Exception as e:
            log.error(f"{symbol}: EMA error - {e}")
            return False

    # ── Orders ────────────────────────────────────────────────────────────────
    def place_limit_buy(self, symbol: str, ask: float, qty: int = ORDER_SIZE):
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        trade = self.ib.placeOrder(contract, LimitOrder("BUY", qty, round(ask, 2)))
        self.ib.sleep(1)
        log.info(f"BUY {symbol} x{qty} @ ${ask:.2f}  orderId={trade.order.orderId}")
        return trade

    def get_avg_fill_price(self, symbol: str, order_id: int) -> float | None:
        self.ib.sleep(2)
        for trade in self.ib.trades():
            if (
                trade.contract.symbol == symbol
                and trade.order.orderId == order_id
                and trade.orderStatus.avgFillPrice > 0
            ):
                avg = float(trade.orderStatus.avgFillPrice)
                log.info(f"{symbol}: avg fill price from broker = ${avg:.4f}")
                return avg

        fills = self.ib.executions()
        symbol_fills = [
            f for f in fills
            if f.contract.symbol == symbol and f.execution.orderId == order_id
        ]
        if symbol_fills:
            total_qty  = sum(f.execution.shares for f in symbol_fills)
            total_cost = sum(f.execution.shares * f.execution.price for f in symbol_fills)
            avg = total_cost / total_qty
            log.info(f"{symbol}: avg fill price from executions = ${avg:.4f}")
            return avg

        log.warning(f"{symbol}: could not retrieve avg fill price for orderId={order_id}")
        return None


# ══════════════════════════════════════════════════════════════════════════════
# MongoDB Helpers
# ══════════════════════════════════════════════════════════════════════════════
def save_trade(symbol: str, limit_price: float, avg_fill_price: float | None,
               qty: int, order_id: int):
    entry_price = avg_fill_price if avg_fill_price else limit_price
    doc = {
        "symbol":          symbol,
        "limit_price":     limit_price,
        "entry_price":     entry_price,
        "avg_fill_price":  avg_fill_price,
        "qty":             qty,
        "order_id":        order_id,
        "status":          "open",
        "entered_at":      datetime.now(timezone.utc),
        "exit_price":      None,
        "exit_reason":     None,
        "exited_at":       None,
        "pnl_pct":         None,
    }
    trades_collection.insert_one(doc)
    ticker_trade_collection.update_one(
        {"symbol": symbol},
        {"$set": {"last_trade": doc, "updated_at": datetime.now(timezone.utc)}},
        upsert=True,
    )
    fill_str = f"${avg_fill_price:.4f}" if avg_fill_price else "pending"
    log.info(f"trade saved  {symbol}  limit=${limit_price:.2f}  avg_fill={fill_str}")


def save_failed_entry(symbol: str, reason: str):
    failed_entries_collection.insert_one({
        "symbol":    symbol,
        "reason":    reason,
        "timestamp": datetime.now(timezone.utc),
    })
    log.info(f"failed entry {symbol} -- {reason}")


# ══════════════════════════════════════════════════════════════════════════════
# Buy Logic
# ══════════════════════════════════════════════════════════════════════════════
def process_buy_signal(symbol: str, ibkr: IBKRBuyClient):
    log.info(f"--- BUY SIGNAL: {symbol} ---")

    if ibkr.has_open_position(symbol):
        save_failed_entry(symbol, "already_in_position"); return
    if ibkr.has_pending_order(symbol):
        save_failed_entry(symbol, "pending_order_exists"); return
    if not ibkr.ema_bullish(symbol):
        save_failed_entry(symbol, "ema_not_bullish"); return

    ask = ibkr.get_ask_price(symbol)
    if not ask:
        save_failed_entry(symbol, "no_ask_price"); return

    trade    = ibkr.place_limit_buy(symbol, ask, ORDER_SIZE)
    order_id = trade.order.orderId
    avg_fill = ibkr.get_avg_fill_price(symbol, order_id)

    save_trade(symbol, ask, avg_fill, ORDER_SIZE, order_id)
    ticker_collection.update_one(
        {"symbol": symbol},
        {"$set": {"last_signal": datetime.now(timezone.utc), "status": "bought"}},
        upsert=True,
    )


# ══════════════════════════════════════════════════════════════════════════════
# Session helpers
# ══════════════════════════════════════════════════════════════════════════════
def load_session_string() -> str:
    if os.path.exists(SESSION_FILE):
        with open(SESSION_FILE, encoding="utf-8") as f:
            return f.read().strip()
    return ""


def save_session_string(session_str: str):
    with open(SESSION_FILE, "w", encoding="utf-8") as f:
        f.write(session_str)
    log.info(f"Session saved -> {SESSION_FILE}")


# ══════════════════════════════════════════════════════════════════════════════
# Telegram Poller
# ══════════════════════════════════════════════════════════════════════════════
async def poll_telegram(ibkr: IBKRBuyClient):
    while True:
        session_str = load_session_string()
        client = TelegramClient(StringSession(session_str), API_ID, API_HASH)
        try:
            await client.start(phone=PHONE)
            save_session_string(client.session.save())

            m = re.search(r't\.me/(?:joinchat/|\+)([A-Za-z0-9_-]+)', INVITE_LINK)
            if not m:
                raise ValueError(f"Cannot parse invite hash: {INVITE_LINK}")
            try:
                await client(ImportChatInviteRequest(m.group(1)))
                log.info("Joined channel")
            except UserAlreadyParticipantError:
                log.info("Already a member")

            channel = await client.get_entity(INVITE_LINK)
            log.info(f"Channel: {channel.title}  (ID: {channel.id})")
            safe_print("-" * 50)

            last_seen_id = None

            while True:
                messages = await client.get_messages(channel, limit=1)

                if messages:
                    msg = messages[0]
                    if msg.text:
                        is_new = (last_seen_id is None or msg.id != last_seen_id)

                        parsed = parse_structured_message(msg.text)
                        if parsed:
                            symbol = parsed["symbol"]
                            status = parsed["status"]
                        else:
                            symbol = extract_ticker(msg.text)
                            status = "NEW" if is_new else "SAME"

                        # Use safe_print for anything that may contain emoji
                        preview = msg.text[:150] + ("..." if len(msg.text) > 150 else "")
                        safe_print(f"\n[{msg.date}]")
                        safe_print(f"   Message : {preview}")
                        safe_print(f"   Ticker  : {symbol or '(not found)'}")
                        safe_print(f"   Status  : {'NEW' if is_new else 'Same as last poll'}")
                        safe_print("-" * 50)

                        if is_new and symbol and is_new_signal(status):
                            log.info(f"Dispatching buy signal for {symbol}")
                            threading.Thread(
                                target=process_buy_signal,
                                args=(symbol, ibkr),
                                daemon=True,
                            ).start()
                        elif is_new and symbol:
                            log.info(f"{symbol}: status='{status}' -- not NEW, skipping.")

                        last_seen_id = msg.id

                log.info(f"Waiting {POLL_INTERVAL}s...")
                await asyncio.sleep(POLL_INTERVAL)

        except FloodWaitError as e:
            log.warning(f"Rate-limited -- waiting {e.seconds}s")
            await asyncio.sleep(e.seconds + 5)
        except KeyboardInterrupt:
            log.info("Stopped by user.")
            break
        except Exception as e:
            log.error(f"Telegram error: {type(e).__name__}: {e}  -- reconnecting in 15s")
            await asyncio.sleep(15)
        finally:
            try:
                await client.disconnect()
            except Exception:
                pass


# ══════════════════════════════════════════════════════════════════════════════
# Entry Point
# ══════════════════════════════════════════════════════════════════════════════
def main():
    log.info("=== Buy Bot Starting ===")
    ibkr = IBKRBuyClient()
    ibkr.connect()
    try:
        asyncio.run(poll_telegram(ibkr))
    except KeyboardInterrupt:
        log.info("Buy bot stopped.")
    finally:
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()

In [ ]:
"""
Buy Bot — Telegram Poller -> IBKR Paper Trading -> MongoDB

HOW THE EVENT LOOP WORKS (ib_insync 0.9.x):
  - util.startLoop() calls nest_asyncio.apply() which PATCHES the current
    event loop to allow nested awaiting — there is NO separate IB thread.
  - Both Telethon and ib_insync share the SAME asyncio event loop.
  - Because of nest_asyncio, ib_insync's sync methods (reqHistoricalData etc.)
    work fine when called from WITHIN an already-running loop.

ROOT CAUSE OF PREVIOUS DEADLOCK:
  - process_buy_signal() ran in a plain threading.Thread.
  - That thread has NO event loop, so ib_insync's sync wrappers (which call
    loop.run_until_complete internally) raised "no running event loop" or hung.

FIX:
  - Replace threading.Thread with asyncio.ensure_future so buy logic runs
    as a coroutine on the shared event loop.
  - All IB calls use the standard async variants (reqHistoricalDataAsync etc.)
    and are awaited normally — no threads, no run_coroutine_threadsafe.
"""

import asyncio
import os
import sys
import re
from datetime import datetime, timezone
import random
import logging

import nest_asyncio
nest_asyncio.apply()   # must be before any asyncio.run() / IB usage

import pandas as pd
from pymongo import MongoClient

from telethon import TelegramClient
from telethon.sessions import StringSession
from telethon.errors import FloodWaitError, UserAlreadyParticipantError
from telethon.tl.functions.messages import ImportChatInviteRequest

from ib_insync import IB, Stock, LimitOrder, util

import warnings
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# Logging — UTF-8 safe (Jupyter + Windows cp1252)
# ══════════════════════════════════════════════════════════════════════════════
class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg  = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh  = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh  = logging.FileHandler("buy_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("buy_bot")
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ══════════════════════════════════════════════════════════════════════════════
# Safe print — handles emoji in raw Telegram message text
# ══════════════════════════════════════════════════════════════════════════════
def safe_print(text: str):
    try:
        encoded = text.encode(sys.stdout.encoding or "utf-8", errors="replace")
        sys.stdout.buffer.write(encoded + b"\n")
        sys.stdout.buffer.flush()
    except AttributeError:
        print(text.encode("utf-8", errors="replace").decode("utf-8", errors="replace"))
    except Exception:
        print(text)


# ══════════════════════════════════════════════════════════════════════════════
# Config
# ══════════════════════════════════════════════════════════════════════════════
API_ID        = 34025130
API_HASH      = "e5e514ebcb8a15042feadfecd7a15cfe"
PHONE         = "+18477675748"
INVITE_LINK   = "https://t.me/+M5HRCxZycqJmOTg5"
POLL_INTERVAL = 20

SESSION_FILE   = "buy_bot_session.txt"
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)
ORDER_SIZE     = 10

mongo_client              = MongoClient("mongodb://localhost:27017/")
db                        = mongo_client["brkout_database"]
trades_collection         = db["trades"]
failed_entries_collection = db["failed_bad_entries"]
ticker_collection         = db["tickers"]
ticker_trade_collection   = db["tickers_trade"]


# ══════════════════════════════════════════════════════════════════════════════
# Telegram Parsers
# ══════════════════════════════════════════════════════════════════════════════
def extract_ticker(message_text: str) -> str | None:
    text = message_text.strip()
    m = re.match(r'^\[.*?\]\s+([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    m = re.match(r'^([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    return None


def parse_structured_message(text: str) -> dict | None:
    result = {"symbol": None, "status": None}
    for line in text.splitlines():
        line = line.strip()
        if re.match(r"(?i)^ticker\s*:", line):
            result["symbol"] = line.split(":", 1)[1].strip().upper()
        if re.match(r"(?i)^status\s*:", line):
            result["status"] = line.split(":", 1)[1].strip()
    if result["symbol"] and result["status"]:
        return result
    return None


def is_new_signal(status_text: str) -> bool:
    return "new" in status_text.lower()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR Client — fully async, runs on shared event loop
# ══════════════════════════════════════════════════════════════════════════════
class IBKRBuyClient:
    def __init__(self):
        self.ib = IB()

    def connect(self):
        # startLoop() applies nest_asyncio so sync IB calls work inside a
        # running loop. connect() itself is a sync blocking call.
        util.startLoop()
        self.ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    # ── sync helpers (safe to call from async context via nest_asyncio) ───────
    def has_open_position(self, symbol: str) -> bool:
        positions = self.ib.positions()
        result = any(p.contract.symbol == symbol and p.position != 0 for p in positions)
        log.debug(f"has_open_position({symbol}) => {result}  (total={len(positions)})")
        return result

    def has_pending_order(self, symbol: str) -> bool:
        open_trades = self.ib.openTrades()
        result = any(t.contract.symbol == symbol for t in open_trades)
        log.debug(f"has_pending_order({symbol}) => {result}  (total={len(open_trades)})")
        return result

    # ── async market data ─────────────────────────────────────────────────────
    async def get_ask_price(self, symbol: str) -> float | None:
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        ticker = self.ib.reqMktData(contract, "106", False, False)
        await asyncio.sleep(3)
        self.ib.cancelMktData(contract)
        log.debug(f"{symbol}: ask={ticker.ask}  last={ticker.last}  "
                  f"close={ticker.close}  bid={ticker.bid}")
        for label, val in [("ask", ticker.ask), ("last", ticker.last), ("close", ticker.close)]:
            if val and float(val) > 0:
                log.info(f"{symbol}: price source='{label}'  value=${float(val):.4f}")
                return float(val)
        log.warning(f"{symbol}: all price fields empty from IBKR")
        return None

    async def get_bars(self, symbol: str, n: int = 70) -> pd.DataFrame | None:
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        for what in ("TRADES", "MIDPOINT"):
            log.debug(f"{symbol}: requesting bars whatToShow={what} useRTH=False")
            raw = await self.ib.reqHistoricalDataAsync(
                contract,
                endDateTime="",
                durationStr="5 D",
                barSizeSetting="5 mins",
                whatToShow=what,
                useRTH=False,
            )
            log.debug(f"{symbol}: bars returned={len(raw) if raw else 0}")
            if raw:
                df = pd.DataFrame([
                    {"high": b.high, "low": b.low, "close": b.close}
                    for b in raw
                ]).tail(n).reset_index(drop=True)
                log.debug(f"{symbol}: df shape={df.shape}  last_close={df['close'].iloc[-1]:.4f}")
                return df
        log.warning(f"{symbol}: no bars returned from IBKR")
        return None

    async def ema_bullish(self, symbol: str) -> bool:
        log.debug(f"ema_bullish({symbol}): fetching bars...")
        try:
            df = await self.get_bars(symbol, n=70)
            if df is None:
                log.warning(f"{symbol}: no bars -- skipping EMA")
                return False
            if len(df) < 50:
                log.warning(f"{symbol}: only {len(df)} bars (need 50) -- skipping EMA")
                return False
            ema21 = df["close"].ewm(span=21, adjust=False).mean().iloc[-1]
            ema50 = df["close"].ewm(span=50, adjust=False).mean().iloc[-1]
            bullish = bool(ema21 > ema50)
            log.info(f"{symbol}: EMA21={ema21:.4f}  EMA50={ema50:.4f}  bullish={bullish}")
            return bullish
        except Exception as e:
            log.error(f"{symbol}: EMA error - {e}")
            return False

    async def place_limit_buy(self, symbol: str, ask: float, qty: int = ORDER_SIZE):
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        order = LimitOrder("BUY", qty, round(ask, 2))
        order.outsideRth = True   # allows pre/post market execution
        trade = self.ib.placeOrder(contract, order)
        await asyncio.sleep(1)
        log.info(f"BUY {symbol} x{qty} @ ${ask:.2f}  "
                 f"orderId={trade.order.orderId}  outsideRth=True  "
                 f"status={trade.orderStatus.status}")
        return trade

    async def get_avg_fill_price(self, symbol: str, order_id: int) -> float | None:
        await asyncio.sleep(2)
        for trade in self.ib.trades():
            if (trade.contract.symbol == symbol
                    and trade.order.orderId == order_id
                    and trade.orderStatus.avgFillPrice > 0):
                avg = float(trade.orderStatus.avgFillPrice)
                log.info(f"{symbol}: avg fill from broker = ${avg:.4f}")
                return avg
        fills = self.ib.executions()
        sf = [f for f in fills
              if f.contract.symbol == symbol and f.execution.orderId == order_id]
        if sf:
            total_qty  = sum(f.execution.shares for f in sf)
            total_cost = sum(f.execution.shares * f.execution.price for f in sf)
            avg = total_cost / total_qty
            log.info(f"{symbol}: avg fill from executions = ${avg:.4f}")
            return avg
        log.warning(f"{symbol}: avg fill not available for orderId={order_id}")
        return None


# ══════════════════════════════════════════════════════════════════════════════
# MongoDB helpers
# ══════════════════════════════════════════════════════════════════════════════
def save_trade(symbol: str, limit_price: float, avg_fill_price: float | None,
               qty: int, order_id: int):
    entry_price = avg_fill_price if avg_fill_price else limit_price
    doc = {
        "symbol":         symbol,
        "limit_price":    limit_price,
        "entry_price":    entry_price,
        "avg_fill_price": avg_fill_price,
        "qty":            qty,
        "order_id":       order_id,
        "status":         "open",
        "entered_at":     datetime.now(timezone.utc),
        "exit_price":     None,
        "exit_reason":    None,
        "exited_at":      None,
        "pnl_pct":        None,
    }
    trades_collection.insert_one(doc)
    ticker_trade_collection.update_one(
        {"symbol": symbol},
        {"$set": {"last_trade": doc, "updated_at": datetime.now(timezone.utc)}},
        upsert=True,
    )
    fill_str = f"${avg_fill_price:.4f}" if avg_fill_price else "pending"
    log.info(f"trade saved  {symbol}  limit=${limit_price:.2f}  avg_fill={fill_str}")


def save_failed_entry(symbol: str, reason: str):
    failed_entries_collection.insert_one({
        "symbol":    symbol,
        "reason":    reason,
        "timestamp": datetime.now(timezone.utc),
    })
    log.info(f"failed entry {symbol} -- {reason}")


# ══════════════════════════════════════════════════════════════════════════════
# Buy Logic — async coroutine (runs on shared event loop, no threads needed)
# ══════════════════════════════════════════════════════════════════════════════
async def process_buy_signal(symbol: str, ibkr: IBKRBuyClient):
    log.info(f"========== BUY SIGNAL: {symbol} ==========")

    # Gate 1 & 2: position / pending order (sync, safe with nest_asyncio)
    if ibkr.has_open_position(symbol):
        save_failed_entry(symbol, "already_in_position"); return
    if ibkr.has_pending_order(symbol):
        save_failed_entry(symbol, "pending_order_exists"); return

    # Gate 3: EMA check (async)
    log.debug(f"{symbol}: GATE 3 - EMA check...")
    if not await ibkr.ema_bullish(symbol):
        save_failed_entry(symbol, "ema_not_bullish"); return
    log.debug(f"{symbol}: GATE 3 PASSED")

    # Gate 4: price (async)
    log.debug(f"{symbol}: GATE 4 - ask price...")
    ask = await ibkr.get_ask_price(symbol)
    if not ask:
        save_failed_entry(symbol, "no_ask_price"); return
    log.debug(f"{symbol}: GATE 4 PASSED  ask=${ask:.4f}")

    # Place order (async)
    trade    = await ibkr.place_limit_buy(symbol, ask, ORDER_SIZE)
    order_id = trade.order.orderId
    avg_fill = await ibkr.get_avg_fill_price(symbol, order_id)

    save_trade(symbol, ask, avg_fill, ORDER_SIZE, order_id)
    ticker_collection.update_one(
        {"symbol": symbol},
        {"$set": {"last_signal": datetime.now(timezone.utc), "status": "bought"}},
        upsert=True,
    )
    log.info(f"========== BUY COMPLETE: {symbol} @ ${ask:.4f} ==========")


# ══════════════════════════════════════════════════════════════════════════════
# Session helpers
# ══════════════════════════════════════════════════════════════════════════════
def load_session_string() -> str:
    if os.path.exists(SESSION_FILE):
        with open(SESSION_FILE, encoding="utf-8") as f:
            return f.read().strip()
    return ""


def save_session_string(session_str: str):
    with open(SESSION_FILE, "w", encoding="utf-8") as f:
        f.write(session_str)
    log.info(f"Session saved -> {SESSION_FILE}")


# ══════════════════════════════════════════════════════════════════════════════
# Telegram Poller
# ══════════════════════════════════════════════════════════════════════════════
async def poll_telegram(ibkr: IBKRBuyClient):
    while True:
        session_str = load_session_string()
        client = TelegramClient(StringSession(session_str), API_ID, API_HASH)
        try:
            await client.start(phone=PHONE)
            save_session_string(client.session.save())

            m = re.search(r't\.me/(?:joinchat/|\+)([A-Za-z0-9_-]+)', INVITE_LINK)
            if not m:
                raise ValueError(f"Cannot parse invite hash: {INVITE_LINK}")
            try:
                await client(ImportChatInviteRequest(m.group(1)))
                log.info("Joined channel")
            except UserAlreadyParticipantError:
                log.info("Already a member")

            channel = await client.get_entity(INVITE_LINK)
            log.info(f"Channel: {channel.title}  (ID: {channel.id})")
            safe_print("-" * 60)

            last_seen_id = None

            while True:
                messages = await client.get_messages(channel, limit=1)
                if messages:
                    msg = messages[0]
                    if msg.text:
                        is_new = (last_seen_id is None or msg.id != last_seen_id)

                        parsed = parse_structured_message(msg.text)
                        if parsed:
                            symbol = parsed["symbol"]
                            status = parsed["status"]
                        else:
                            symbol = extract_ticker(msg.text)
                            status = "NEW" if is_new else "SAME"

                        preview = msg.text[:150] + ("..." if len(msg.text) > 150 else "")
                        safe_print(f"\n[{msg.date}]")
                        safe_print(f"   Message : {preview}")
                        safe_print(f"   Ticker  : {symbol or '(not found)'}")
                        safe_print(f"   Status  : {'NEW' if is_new else 'Same as last poll'}")
                        log.debug(f"is_new={is_new}  symbol={symbol}  status='{status}'")
                        safe_print("-" * 60)

                        if is_new and symbol and is_new_signal(status):
                            log.info(f"Dispatching buy coroutine for {symbol}")
                            # Schedule as a fire-and-forget task on the shared loop
                            asyncio.ensure_future(process_buy_signal(symbol, ibkr))
                        elif is_new and symbol:
                            log.info(f"{symbol}: status='{status}' -- not NEW, skipping.")

                        last_seen_id = msg.id

                log.info(f"Waiting {POLL_INTERVAL}s...")
                await asyncio.sleep(POLL_INTERVAL)

        except FloodWaitError as e:
            log.warning(f"Rate-limited -- waiting {e.seconds}s")
            await asyncio.sleep(e.seconds + 5)
        except KeyboardInterrupt:
            log.info("Stopped by user.")
            break
        except Exception as e:
            log.error(f"Telegram error: {type(e).__name__}: {e}  -- reconnecting in 15s")
            await asyncio.sleep(15)
        finally:
            try:
                await client.disconnect()
            except Exception:
                pass


# ══════════════════════════════════════════════════════════════════════════════
# Entry Point
# ══════════════════════════════════════════════════════════════════════════════
def main():
    log.info("=== Buy Bot Starting ===")
    ibkr = IBKRBuyClient()
    ibkr.connect()
    try:
        asyncio.run(poll_telegram(ibkr))
    except KeyboardInterrupt:
        log.info("Buy bot stopped.")
    finally:
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()


2026-03-17 07:43:03,244 [INFO] === Buy Bot Starting ===
2026-03-17 07:43:03,400 [INFO] IBKR connected | accounts: ['DUD087866']
2026-03-17 07:43:13,779 [INFO] Session saved -> buy_bot_session.txt
2026-03-17 07:43:13,818 [INFO] Already a member
2026-03-17 07:43:13,851 [INFO] Channel: StockMembers | News, Alerts & Updates  (ID: 1164489927)
------------------------------------------------------------

[2026-03-17 11:10:26+00:00]
   Message : PMAX volume spike .83
   Ticker  : PMAX
   Status  : NEW
2026-03-17 07:43:13,891 [DEBUG] is_new=True  symbol=PMAX  status='NEW'
------------------------------------------------------------
2026-03-17 07:43:13,891 [INFO] Dispatching buy coroutine for PMAX
2026-03-17 07:43:13,892 [INFO] Waiting 20s...
2026-03-17 07:43:13,893 [INFO] ========== BUY SIGNAL: PMAX ==========
2026-03-17 07:43:13,896 [DEBUG] has_open_position(PMAX) => True  (total=350)
2026-03-17 07:43:13,902 [INFO] failed entry PMAX -- already_in_position

[2026-03-17 11:10:26+00:00]
   Messa

Error 10349, reqId 8: Order TIF was set to GTC based on order preset.
Canceled order: Trade(contract=Stock(conId=799578422, symbol='CREG', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='CREG', tradingClass='SCM'), order=LimitOrder(orderId=8, clientId=686, action='BUY', totalQuantity=10, lmtPrice=1.58, outsideRth=True), orderStatus=OrderStatus(orderId=8, status='Cancelled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 17, 12, 37, 18, 27274, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0), TradeLogEntry(time=datetime.datetime(2026, 3, 17, 12, 37, 18, 32426, tzinfo=datetime.timezone.utc), status='Cancelled', message='Error 10349, reqId 8: Order TIF was set to GTC based on order preset.', errorCode=10349)], advancedError='')


2026-03-17 08:37:19,040 [INFO] BUY CREG x10 @ $1.58  orderId=8  outsideRth=True  status=Filled
2026-03-17 08:37:21,046 [INFO] CREG: avg fill from broker = $1.5800
2026-03-17 08:37:21,051 [INFO] trade saved  CREG  limit=$1.58  avg_fill=$1.5800
2026-03-17 08:37:21,053 [INFO] ========== BUY COMPLETE: CREG @ $1.5800 ==========

[2026-03-17 12:36:54+00:00]
   Message : CREG volume increased 1.66
   Ticker  : CREG
   Status  : Same as last poll
2026-03-17 08:37:34,524 [DEBUG] is_new=False  symbol=CREG  status='SAME'
------------------------------------------------------------
2026-03-17 08:37:34,525 [INFO] Waiting 20s...

[2026-03-17 12:36:54+00:00]
   Message : CREG volume increased 1.66
   Ticker  : CREG
   Status  : Same as last poll
2026-03-17 08:37:54,569 [DEBUG] is_new=False  symbol=CREG  status='SAME'
------------------------------------------------------------
2026-03-17 08:37:54,570 [INFO] Waiting 20s...

[2026-03-17 12:36:54+00:00]
   Message : CREG volume increased 1.66
   Ticker 

Server closed the connection: [WinError 10054] An existing connection was forcibly closed by the remote host
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Attempt 1 at connecting failed: TimeoutError: 
Attempt 2 at connecting failed: TimeoutError: 
Attempt 3 at connecting failed: TimeoutError: 
Attempt 4 at connecting failed: ConnectionAbortedError: [Errno 10053] Connect call failed ('149.154.175.58', 443)
Attempt 5 at connecting failed: OSError: [Errno 10051] Connect call failed ('149.154.175.58', 443)
Attempt 6 at connecting failed: OSError: [Errno 10051] Connect call failed ('149.154.175.58', 443)
Attempt 1 at connecting failed: OSError: [Errno 10051] Connect call failed ('149.154.175.58', 443)
Attempt 2 at connecting failed: OSError: [Errno 10051] Connect call failed ('149.154.175.58', 443)
Attempt 3 at connecting failed: OSError: [Errno 10051] Connect call failed ('149.154.175.58', 443)



[2026-03-17 13:02:37+00:00]
   Message : CXAI - CXAI Launches AI-Powered Zero-Touch Campus Deployment Platform for Enterprise Workplaces
   Ticker  : CXAI
   Status  : Same as last poll
2026-03-17 09:12:16,086 [DEBUG] is_new=False  symbol=CXAI  status='SAME'
------------------------------------------------------------
2026-03-17 09:12:16,087 [INFO] Waiting 20s...

[2026-03-17 13:02:37+00:00]
   Message : CXAI - CXAI Launches AI-Powered Zero-Touch Campus Deployment Platform for Enterprise Workplaces
   Ticker  : CXAI
   Status  : Same as last poll
2026-03-17 09:12:36,138 [DEBUG] is_new=False  symbol=CXAI  status='SAME'
------------------------------------------------------------
2026-03-17 09:12:36,138 [INFO] Waiting 20s...


...


[2026-03-17 14:58:35+00:00]
   Message : CREG new high of the day 2.41 🤑
   Ticker  : CREG
   Status  : Same as last poll
2026-03-17 11:14:11,629 [DEBUG] is_new=False  symbol=CREG  status='SAME'
------------------------------------------------------------
2026-03-17 11:14:11,630 [INFO] Waiting 20s...

[2026-03-17 14:58:35+00:00]
   Message : CREG new high of the day 2.41 🤑
   Ticker  : CREG
   Status  : Same as last poll
2026-03-17 11:14:35,571 [DEBUG] is_new=False  symbol=CREG  status='SAME'
------------------------------------------------------------
2026-03-17 11:14:35,572 [INFO] Waiting 20s...

[2026-03-17 14:58:35+00:00]
   Message : CREG new high of the day 2.41 🤑
   Ticker  : CREG
   Status  : Same as last poll
2026-03-17 11:14:55,636 [DEBUG] is_new=False  symbol=CREG  status='SAME'
------------------------------------------------------------
2026-03-17 11:14:55,637 [INFO] Waiting 20s...

[2026-03-17 14:58:35+00:00]
   Message : CREG new high of the day 2.41 🤑
   Ticker  : CREG

Server closed the connection: [WinError 10054] An existing connection was forcibly closed by the remote host



[2026-03-17 15:28:08+00:00]
   Message : SWMR halted up 18.15
   Ticker  : SWMR
   Status  : Same as last poll
2026-03-17 11:33:42,111 [DEBUG] is_new=False  symbol=SWMR  status='SAME'
------------------------------------------------------------
2026-03-17 11:33:42,112 [INFO] Waiting 20s...

[2026-03-17 15:28:08+00:00]
   Message : SWMR halted up 18.15
   Ticker  : SWMR
   Status  : Same as last poll
2026-03-17 11:34:02,154 [DEBUG] is_new=False  symbol=SWMR  status='SAME'
------------------------------------------------------------
2026-03-17 11:34:02,154 [INFO] Waiting 20s...

[2026-03-17 15:28:08+00:00]
   Message : SWMR halted up 18.15
   Ticker  : SWMR
   Status  : Same as last poll
2026-03-17 11:34:22,195 [DEBUG] is_new=False  symbol=SWMR  status='SAME'
------------------------------------------------------------
2026-03-17 11:34:22,196 [INFO] Waiting 20s...

[2026-03-17 15:28:08+00:00]
   Message : SWMR halted up 18.15
   Ticker  : SWMR
   Status  : Same as last poll
2026-03-17 1

Error 10349, reqId 64: Order TIF was set to GTC based on order preset.
Canceled order: Trade(contract=Stock(conId=659057942, symbol='EDSA', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='EDSA', tradingClass='SCM'), order=LimitOrder(orderId=64, clientId=686, action='BUY', totalQuantity=10, lmtPrice=7.84, outsideRth=True), orderStatus=OrderStatus(orderId=64, status='Cancelled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 17, 15, 43, 26, 953345, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0), TradeLogEntry(time=datetime.datetime(2026, 3, 17, 15, 43, 26, 956508, tzinfo=datetime.timezone.utc), status='Cancelled', message='Error 10349, reqId 64: Order TIF was set to GTC based on order preset.', errorCode=10349)], advancedError='')


2026-03-17 11:43:27,966 [INFO] BUY EDSA x10 @ $7.84  orderId=64  outsideRth=True  status=Submitted
2026-03-17 11:43:29,977 [INFO] EDSA: avg fill from broker = $7.8000
2026-03-17 11:43:29,980 [INFO] trade saved  EDSA  limit=$7.84  avg_fill=$7.8000
2026-03-17 11:43:29,982 [INFO] ========== BUY COMPLETE: EDSA @ $7.8400 ==========

[2026-03-17 15:43:22+00:00]
   Message : EDSA volume increased 7.66
   Ticker  : EDSA
   Status  : Same as last poll
2026-03-17 11:43:43,484 [DEBUG] is_new=False  symbol=EDSA  status='SAME'
------------------------------------------------------------
2026-03-17 11:43:43,485 [INFO] Waiting 20s...

[2026-03-17 15:43:22+00:00]
   Message : EDSA volume increased 7.66
   Ticker  : EDSA
   Status  : Same as last poll
2026-03-17 11:44:03,521 [DEBUG] is_new=False  symbol=EDSA  status='SAME'
------------------------------------------------------------
2026-03-17 11:44:03,523 [INFO] Waiting 20s...
